### Setup

In [7]:
question_type = "multiple-choice" # Options: "multiple-choice", "open-ended"
dataset = "ptpt" # Options: "ptpt", "ptbr"
prompt_language = dataset # Options: "ptpt", "ptbr", "en"
models = {"claude-haiku-4.5", "deepseek-chat-v3.1", "gemini-2.5-flash", "gemma-3-27b-it", "gpt-5","llama-3.3-70b-instruct", "qwen3-8b", "qwen3-14b", "qwen3-32b", "qwen3-30b-a3b", "qwen3-235b-a22b"}

### Load Golden Answers

In [8]:
import json

with open(f'data/{dataset}-{question_type}-qa-pairs.json', 'r', encoding='utf-8') as f:
    golden_qa_pairs = json.load(f)

### Load Model Answers

In [9]:
# Initialize a dict instead of a list
responses = {model: [] for model in models}

for model in models:
    responses_file = f'{model}-{dataset}-responses-prompt-language-{prompt_language}.json'
    with open(f'results/{question_type}/{responses_file}', 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  # skip empty lines
                responses[model].append(json.loads(line))
print(f'Loaded responses for models: {list(responses.keys())}')

Loaded responses for models: ['qwen3-32b', 'qwen3-8b', 'qwen3-14b', 'deepseek-chat-v3.1', 'claude-haiku-4.5', 'gemma-3-27b-it', 'qwen3-235b-a22b', 'llama-3.3-70b-instruct', 'gemini-2.5-flash', 'qwen3-30b-a3b', 'gpt-5']


### Parse Final Answers

In [10]:
import re

ids_answered_by_all = {item["id"] for item in golden_qa_pairs}

print(f"Number of golden QA pairs: {len(ids_answered_by_all)}")

for model in models:
    for response in responses[model]:
        try:
            text = response["raw_response"]["choice.message.content"]
        except KeyError:
            # skip responses without the content field
            response['final_answer'] = None
            ids_answered_by_all.discard(response["id"])
            continue
        m = re.search(r'(?<=\\boxed\{)([A-Za-z])(?=\}(?!.*\\boxed))', text)
        if m:
            response["final_answer"] = m.group(0)   # store the letter string, not the match object
        else:
            response["final_answer"] = None
            ids_answered_by_all.discard(response["id"])

print(f"Answered by all models after filtering: {len(ids_answered_by_all)}")

Number of golden QA pairs: 419
Answered by all models after filtering: 158


### Keep Only Questions Answered by all models

In [11]:
""" 
for pair in golden_qa_pairs:
    if pair["id"] not in ids_answered_by_all:
        golden_qa_pairs.remove(pair)
for model in models:
    responses[model] = [resp for resp in responses[model] if resp["id"] in ids_answered_by_all]
"""

' \nfor pair in golden_qa_pairs:\n    if pair["id"] not in ids_answered_by_all:\n        golden_qa_pairs.remove(pair)\nfor model in models:\n    responses[model] = [resp for resp in responses[model] if resp["id"] in ids_answered_by_all]\n'

### Calculate accuracies

In [12]:
from collections import defaultdict

# Build golden lookup by question id
golden_by_id = {item["id"]: item for item in golden_qa_pairs}

# Helper to initialize stats dict
def init_stats():
    return {model: {lvl: {"correct": 0, "total": 0} for lvl in range(1, 5)} for model in models}

# We keep figure/no-figure splits plus overall
stats_with_fig = init_stats()
stats_no_fig = init_stats()
stats_all = init_stats()  # global (figure + no figure)

for model in models:
    for r in responses.get(model, []):
        qid = r.get("id")
        g = golden_by_id.get(qid)
        if g is None:
            continue
        # Level handling (ensure integer 1..4)
        lvl_raw = g.get("level", 0)
        try:
            lvl = int(lvl_raw)
        except Exception:
            continue
        if lvl not in (1, 2, 3, 4):
            continue

        # Figure presence (any of the possible fields)
        contains_figure = bool(
            g.get("contains_latex_figure_in_question")
        )

        final = r.get("final_answer")
        correct_opt = g.get("correct_option")
        is_correct = (
            final is not None
            and correct_opt is not None
            and str(final).strip().upper() == str(correct_opt).strip().upper()
        )

        # Always update global stats
        stats_all[model][lvl]["total"] += 1
        if is_correct:
            stats_all[model][lvl]["correct"] += 1

        # Split by figure presence
        if contains_figure:
            stats_with_fig[model][lvl]["total"] += 1
            if is_correct:
                stats_with_fig[model][lvl]["correct"] += 1
        else:
            stats_no_fig[model][lvl]["total"] += 1
            if is_correct:
                stats_no_fig[model][lvl]["correct"] += 1


def compute_acc(stats):
    out = {}
    for model in models:
        out[model] = {}
        for lvl in range(1, 5):
            c = stats[model][lvl]["correct"]
            t = stats[model][lvl]["total"]
            out[model][lvl] = {
                "correct": c,
                "total": t,
                "accuracy": (c / t * 100.0) if t else None,
            }
    return out


acc_with_fig = compute_acc(stats_with_fig)
acc_no_fig = compute_acc(stats_no_fig)
acc_all = compute_acc(stats_all)  # global


def format_acc(title, acc):
    lines = [f"=== {title} ==="]
    for model in sorted(models):
        lines.append("")
        lines.append(f"Model: {model}")
        for lvl in range(1, 5):
            a = acc[model][lvl]
            t = a["total"]
            if not t:
                lines.append(f"  Level {lvl}: no data")
            else:
                lines.append(
                    f"  Level {lvl}: {a['correct']}/{t} correct ({a['accuracy']:.2f}%)"
                )
    lines.append("")
    return "\n".join(lines)


report_text = []
report_text.append(format_acc("Overall (fig and no fig)", acc_all))
report_text.append(format_acc("With figures only", acc_with_fig))
report_text.append(format_acc("Without figures", acc_no_fig))
report_text = "\n".join(report_text)

# Save to results directory with requested filename pattern
out_filename = f"{dataset}-{question_type}-all-models-prompt-language-{prompt_language}-acc-report.txt"
out_path = f"results/accuracy-reports/{out_filename}"
with open(out_path, "w", encoding="utf-8") as out:
    out.write(report_text)